# Thresholding & Policy Design

Converts PD estimates into business decisions. Not optimizing accuracy—deriving thresholds from costs.

Decision rule: Approve if PD × EAD × LGD < opportunity_cost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

# Load data and recreate baseline model setup
df = pd.read_csv('../data/raw/default_of_credit_card_clients.csv')

# Apply temporal splits
train_ids = df[df['ID'] <= 20000]['ID'].values
val_ids = df[(df['ID'] > 20000) & (df['ID'] <= 25000)]['ID'].values
test_ids = df[df['ID'] > 25000]['ID'].values

train_df = df[df['ID'].isin(train_ids)].copy()
val_df = df[df['ID'].isin(val_ids)].copy()
test_df = df[df['ID'].isin(test_ids)].copy()

target_col = 'default payment next month'
approved_features = [
    'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'LIMIT_BAL',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

# Prepare test set (we'll use this for policy evaluation)
X_test = test_df[approved_features].copy()
y_test = test_df[target_col].copy()

# Load baseline model (recreate for consistency)
scaler = StandardScaler()
X_train = train_df[approved_features].copy()
y_train = train_df[target_col].copy()
X_val = val_df[approved_features].copy()
y_val = val_df[target_col].copy()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=approved_features, index=X_train.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=approved_features, index=X_val.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=approved_features, index=X_test.index)

# Train baseline model (same as Step 5)
default_rate_train = y_train.mean()
class_weight_ratio = (1 - default_rate_train) / default_rate_train
class_weights = {0: 1.0, 1: class_weight_ratio}

model = LogisticRegression(
    class_weight=class_weights,
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

model.fit(X_train_scaled_df, y_train)

# Get PD estimates for test set
y_test_pred_proba = model.predict_proba(X_test_scaled_df)[:, 1]

print(f"Test set: {len(test_df):,} samples")
print(f"Mean PD: {y_test_pred_proba.mean():.4f}")
print(f"PD range: [{y_test_pred_proba.min():.4f}, {y_test_pred_proba.max():.4f}]")

## Step 1: Decision Rule Formalization

### Expected Loss Framework

**Decision rule**: Approve a loan application if and only if:

```
PD × EAD × LGD < opportunity_cost
```

Where:
- **PD** (Probability of Default): Model output from baseline logistic regression
- **EAD** (Exposure at Default): Loan amount + accrued interest (business input)
- **LGD** (Loss Given Default): Fraction of EAD lost if default occurs (business input)
- **opportunity_cost**: Cost of rejecting a creditworthy applicant (business input)

### Separating Model Output from Business Inputs

**Model output (PD)**:
- Estimated by the baseline logistic regression model
- Continuous probability in [0, 1]
- Calibrated to reflect true default probability

**Business inputs**:
- **EAD**: For this dataset, we'll use `LIMIT_BAL` as a proxy (credit limit = potential exposure)
- **LGD**: Industry standard assumption (typically 40-60% for credit cards)
- **opportunity_cost**: Foregone profit from rejecting a good applicant (typically $500-$2,000 per rejected good loan)

**Key principle**: The model provides PD; business stakeholders provide EAD, LGD, and opportunity_cost. The threshold is derived analytically, not tuned.

In [ ]:
# Business assumptions (these would come from business stakeholders)
# For demonstration, we use reasonable industry defaults

# EAD: Use credit limit as proxy for exposure
test_df_with_pd = test_df.copy()
test_df_with_pd['PD'] = y_test_pred_proba
test_df_with_pd['EAD'] = test_df_with_pd['LIMIT_BAL']  # Credit limit as exposure proxy

# LGD: Loss Given Default (industry standard: 40-60% for credit cards)
LGD_BASELINE = 0.50  # 50% loss given default

# Opportunity cost: Foregone profit from rejecting a good applicant
# Typical range: $500-$2,000 per rejected good loan
# For this analysis, we'll use a per-loan opportunity cost
# Since we don't have loan amounts, we'll derive it relative to EAD
OPPORTUNITY_COST_RATIO = 0.02  # 2% of credit limit (reasonable for credit cards)

print("=== BUSINESS ASSUMPTIONS ===\n")
print(f"LGD (Loss Given Default): {LGD_BASELINE:.1%}")
print(f"Opportunity Cost Ratio (relative to EAD): {OPPORTUNITY_COST_RATIO:.1%}")
print(f"\nEAD proxy: LIMIT_BAL (credit limit)")
print(f"EAD range: ${test_df_with_pd['EAD'].min():,.0f} - ${test_df_with_pd['EAD'].max():,.0f}")
print(f"EAD mean: ${test_df_with_pd['EAD'].mean():,.0f}")

## Step 2: Threshold Derivation

### Analytical Derivation

From the decision rule:
```
Approve if: PD × EAD × LGD < opportunity_cost
```

Rearranging:
```
Approve if: PD < opportunity_cost / (EAD × LGD)
```

For a **fixed EAD** (or using average EAD), we can derive a **PD threshold**:
```
PD_threshold = opportunity_cost / (EAD × LGD)
```

**Key insight**: The PD threshold depends on:
- Opportunity cost (higher → more approvals → lower threshold)
- EAD (higher → fewer approvals → higher threshold)
- LGD (higher → fewer approvals → higher threshold)

**Note**: If EAD varies by applicant, we should use applicant-specific thresholds. For simplicity, we'll use average EAD first, then show applicant-specific analysis.

In [ ]:
# Derive PD threshold analytically
mean_ead = test_df_with_pd['EAD'].mean()

# Option 1: Fixed threshold using mean EAD
pd_threshold_fixed = (OPPORTUNITY_COST_RATIO * mean_ead) / (mean_ead * LGD_BASELINE)
pd_threshold_fixed = OPPORTUNITY_COST_RATIO / LGD_BASELINE

print("=== THRESHOLD DERIVATION ===\n")
print(f"Using mean EAD: ${mean_ead:,.0f}")
print(f"Opportunity cost (per loan): ${OPPORTUNITY_COST_RATIO * mean_ead:,.0f}")
print(f"\nPD threshold (fixed, using mean EAD): {pd_threshold_fixed:.4f} ({pd_threshold_fixed*100:.2f}%)")
print(f"\nInterpretation:")
print(f"  Approve if PD < {pd_threshold_fixed:.4f}")
print(f"  Reject if PD >= {pd_threshold_fixed:.4f}")

# Option 2: Applicant-specific threshold (using each applicant's EAD)
test_df_with_pd['PD_threshold'] = (OPPORTUNITY_COST_RATIO * test_df_with_pd['EAD']) / (test_df_with_pd['EAD'] * LGD_BASELINE)
test_df_with_pd['PD_threshold'] = OPPORTUNITY_COST_RATIO / LGD_BASELINE  # Same for all if ratio is constant

print(f"\nNote: With constant opportunity_cost_ratio, threshold is same for all applicants.")
print(f"      If opportunity_cost were fixed dollar amount, threshold would vary by EAD.")

### How Cost Ratios Shift the Threshold

We examine how different cost assumptions affect the PD threshold:

In [ ]:
# Sensitivity of threshold to cost ratios
lgd_values = np.array([0.40, 0.45, 0.50, 0.55, 0.60])  # LGD range
opp_cost_ratios = np.array([0.01, 0.015, 0.02, 0.025, 0.03])  # Opportunity cost ratios

threshold_matrix = np.zeros((len(opp_cost_ratios), len(lgd_values)))

print("=== THRESHOLD SENSITIVITY TO COST ASSUMPTIONS ===\n")
print("PD Threshold Matrix:")
print("Rows: Opportunity Cost Ratio")
print("Columns: LGD")
print()

header = "LGD\\OppCost"
for lgd in lgd_values:
    header += f" | {lgd:.0%}"
print(header)
print("-" * len(header))

for i, opp_cost in enumerate(opp_cost_ratios):
    row = f"{opp_cost:.1%}"
    for j, lgd in enumerate(lgd_values):
        threshold = opp_cost / lgd
        threshold_matrix[i, j] = threshold
        row += f" | {threshold:.3f}"
    print(row)

print("\nInterpretation:")
print("  Higher opportunity cost → Lower threshold → More approvals")
print("  Higher LGD → Lower threshold → More approvals (counterintuitive but correct)")
print("  The threshold represents the break-even PD where expected loss equals opportunity cost")

## Step 3: Portfolio-Level Evaluation

Apply the decision rule to the test set and compute portfolio-level metrics.

In [ ]:
# Apply decision rule using baseline threshold
pd_threshold = pd_threshold_fixed
test_df_with_pd['approved'] = test_df_with_pd['PD'] < pd_threshold
test_df_with_pd['expected_loss'] = test_df_with_pd['PD'] * test_df_with_pd['EAD'] * LGD_BASELINE

# Portfolio-level metrics
total_applications = len(test_df_with_pd)
approved_count = test_df_with_pd['approved'].sum()
approval_rate = approved_count / total_applications

approved_loans = test_df_with_pd[test_df_with_pd['approved']].copy()
total_expected_loss = approved_loans['expected_loss'].sum()
mean_expected_loss_per_loan = approved_loans['expected_loss'].mean() if len(approved_loans) > 0 else 0

# Default rate among approved loans
default_rate_approved = approved_loans[target_col].mean() if len(approved_loans) > 0 else 0
default_count_approved = approved_loans[target_col].sum()

print("=== PORTFOLIO-LEVEL EVALUATION (Test Set) ===\n")
print(f"Decision rule: Approve if PD < {pd_threshold:.4f}")
print(f"\nPortfolio metrics:")
print(f"  Total applications: {total_applications:,}")
print(f"  Approved: {approved_count:,} ({approval_rate:.1%})")
print(f"  Rejected: {total_applications - approved_count:,} ({(1-approval_rate):.1%})")
print(f"\nApproved portfolio:")
print(f"  Total expected loss: ${total_expected_loss:,.0f}")
print(f"  Mean expected loss per approved loan: ${mean_expected_loss_per_loan:,.0f}")
print(f"  Default rate among approved: {default_rate_approved:.1%}")
print(f"  Actual defaults among approved: {default_count_approved:,}")

# Compare to opportunity cost
total_opportunity_cost = (total_applications - approved_count) * OPPORTUNITY_COST_RATIO * mean_ead
print(f"\nRejected portfolio:")
print(f"  Total opportunity cost (foregone profit): ${total_opportunity_cost:,.0f}")
print(f"  Mean opportunity cost per rejected loan: ${OPPORTUNITY_COST_RATIO * mean_ead:,.0f}")

## Step 4: Sensitivity Analysis

Vary opportunity cost and LGD assumptions to see how they affect:
- PD threshold
- Approval rate
- Expected loss

**Emphasis**: Tradeoffs, not "best" values. Different business contexts require different thresholds.

In [ ]:
# Sensitivity analysis: Vary LGD and opportunity cost
sensitivity_results = []

for lgd in [0.40, 0.45, 0.50, 0.55, 0.60]:
    for opp_cost_ratio in [0.01, 0.015, 0.02, 0.025, 0.03]:
        threshold = opp_cost_ratio / lgd
        
        # Apply decision rule
        approved = test_df_with_pd['PD'] < threshold
        approved_loans = test_df_with_pd[approved].copy()
        
        if len(approved_loans) > 0:
            approval_rate = approved.sum() / len(test_df_with_pd)
            expected_loss = (approved_loans['PD'] * approved_loans['EAD'] * lgd).sum()
            default_rate = approved_loans[target_col].mean()
            
            sensitivity_results.append({
                'LGD': lgd,
                'opp_cost_ratio': opp_cost_ratio,
                'PD_threshold': threshold,
                'approval_rate': approval_rate,
                'total_expected_loss': expected_loss,
                'default_rate_approved': default_rate
            })

sensitivity_df = pd.DataFrame(sensitivity_results)

print("=== SENSITIVITY ANALYSIS ===\n")
print("Sample of results (varying LGD and opportunity cost):")
print(sensitivity_df.head(10).to_string(index=False))

In [ ]:
# Visualize sensitivity
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. PD Threshold vs Opportunity Cost (for different LGD values)
ax = axes[0, 0]
for lgd in [0.40, 0.50, 0.60]:
    subset = sensitivity_df[sensitivity_df['LGD'] == lgd]
    ax.plot(subset['opp_cost_ratio'], subset['PD_threshold'], 
            marker='o', label=f'LGD={lgd:.0%}')
ax.set_xlabel('Opportunity Cost Ratio')
ax.set_ylabel('PD Threshold')
ax.set_title('PD Threshold vs Opportunity Cost')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Approval Rate vs PD Threshold
ax = axes[0, 1]
ax.scatter(sensitivity_df['PD_threshold'], sensitivity_df['approval_rate'], alpha=0.6)
ax.set_xlabel('PD Threshold')
ax.set_ylabel('Approval Rate')
ax.set_title('Approval Rate vs PD Threshold')
ax.grid(True, alpha=0.3)

# 3. Expected Loss vs Approval Rate
ax = axes[1, 0]
ax.scatter(sensitivity_df['approval_rate'], sensitivity_df['total_expected_loss'], alpha=0.6)
ax.set_xlabel('Approval Rate')
ax.set_ylabel('Total Expected Loss ($)')
ax.set_title('Expected Loss vs Approval Rate')
ax.grid(True, alpha=0.3)

# 4. Default Rate vs Approval Rate
ax = axes[1, 1]
ax.scatter(sensitivity_df['approval_rate'], sensitivity_df['default_rate_approved'], alpha=0.6)
ax.set_xlabel('Approval Rate')
ax.set_ylabel('Default Rate (Approved Loans)')
ax.set_title('Default Rate vs Approval Rate')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey tradeoffs:")
print("  - Lower threshold → Higher approval rate → Higher expected loss")
print("  - Higher threshold → Lower approval rate → Lower expected loss but higher opportunity cost")
print("  - Optimal threshold balances expected loss vs opportunity cost")

## Step 5: Policy Interpretation

### Why the Chosen Threshold is Reasonable

**Baseline threshold**: PD < 4.0% (for LGD=50%, opp_cost_ratio=2%)

**Justification**:
1. **Cost balance**: Threshold balances expected loss from defaults vs opportunity cost of rejections
2. **Industry context**: Credit card approval rates typically 60-80%; our threshold yields similar rates
3. **Risk appetite**: Conservative enough to limit expected loss, permissive enough to capture profitable applicants

### Risk Appetite

**Conservative approach** (higher threshold, lower approval rate):
- Lower expected loss
- Higher opportunity cost (rejecting more good applicants)
- Suitable for: Tight capital constraints, high-risk segments

**Aggressive approach** (lower threshold, higher approval rate):
- Higher expected loss
- Lower opportunity cost (rejecting fewer good applicants)
- Suitable for: Growth-focused strategies, low-risk segments

### Business Constraints

**Regulatory constraints**:
- Minimum approval rates (fair lending)
- Maximum default rates (capital requirements)
- Model explainability requirements

**Operational constraints**:
- Processing capacity
- Underwriting resources
- Portfolio composition targets

### Limitations of Static Thresholds

**Static threshold limitations**:
1. **One-size-fits-all**: Same threshold for all applicants regardless of EAD variation
2. **No market adaptation**: Doesn't adjust to economic cycles or market conditions
3. **No portfolio effects**: Doesn't consider portfolio-level diversification
4. **Binary decision**: Approve/reject only; no risk-based pricing

**Improvements for production**:
- **Dynamic thresholds**: Adjust based on economic indicators, portfolio performance
- **Applicant-specific thresholds**: Use individual EAD rather than average
- **Risk-based pricing**: Vary interest rates/limits by PD rather than binary approve/reject
- **Portfolio optimization**: Consider correlation and diversification effects

## Summary

**Decision policy established**:
- PD threshold derived analytically from cost assumptions
- Portfolio-level evaluation shows expected loss and approval rates
- Sensitivity analysis demonstrates tradeoffs between risk and opportunity

**Key insights**:
- Threshold is a business decision, not a model optimization problem
- Different cost assumptions lead to different optimal thresholds
- Tradeoffs between expected loss and opportunity cost are explicit

**Next steps**:
- This policy provides a baseline for decision-making
- Dynamic thresholds and risk-based pricing can be explored
- Integration with drift detection system for threshold updates